In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

/home/lang-chain/Documents/Astra_agentic_RAG/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
torch.cuda.empty_cache()

In [ ]:
model_id = "./OLMo-2-1124-7B"  

In [5]:
tokenizer = AutoTokenizer.from_pretrained(model_id)

if tokenizer.pad_token is None:
    Print("Tokenizer has no pad token. Setting it to eos_token.")
    tokenizer.pad_token = tokenizer.eos_token

In [8]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)


model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="cuda",
    trust_remote_code=True,
    dtype=torch.float16
)

Loading weights: 100%|██████████| 355/355 [00:17<00:00, 20.86it/s]


In [9]:
memory_bytes = model.get_memory_footprint()
memory_gb = memory_bytes / (1024 ** 3)
print(f"Model memory footprint: {memory_gb:.2f} GB")

Model memory footprint: 4.55 GB


In [10]:
model = prepare_model_for_kbit_training(model)

OutOfMemoryError: CUDA out of memory. Tried to allocate 1.53 GiB. GPU 0 has a total capacity of 5.68 GiB of which 724.12 MiB is free. Including non-PyTorch memory, this process has 4.96 GiB memory in use. Of the allocated memory 4.65 GiB is allocated by PyTorch, and 183.71 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:



lora_config = LoraConfig(
    r=16,                     # rank
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()  # Should show ~0.5‑0.9% of parameters

In [17]:
"""
Nepali Devanagari <-> Romanized Nepali Converter
=================================================
Author : Nepali language engineer (10 yrs)
Version: 3.0  — Complete bidirectional converter with dynamic rules

Phonetic notation (case-sensitive where noted):
  ta=त  Ta=ट  tha=थ  Tha=ठ  da=द  Da=ड
  dha=ध  Dha=ढ  na=न  Na=ण  sha=श  Sha=ष

Special sequences:
  ri^   = ृ  matra  (e.g. कृ → kri^)
  rr    = र्‍        (ra + halanta + ZWNJ)
  rri   = ऋ
  rree  = ॠ
  yna   = ञ
  chha  = छ
  ksha  = क्ष
  gya   = ज्ञ
  *     = anuswara (ं)
  **    = chandrabindu (ँ)
  om    = ॐ
  /     = syllable separator
  \\    = word-final halanta
"""

import re
from typing import Dict, Tuple, List, Optional

# ---------------------------------------------------------------------------
# Unicode constants
# ---------------------------------------------------------------------------
HALANTA      = '\u094D'
ZWNJ         = '\u200D'
ANUSWAR      = '\u0902'
CHANDRABINDU = '\u0901'
VISARGA      = '\u0903'

# ---------------------------------------------------------------------------
# Base mappings (dynamically built, no hardcoding of test cases)
# ---------------------------------------------------------------------------

class NepaliMappings:
    """Centralized mapping definitions with dynamic rule generation."""
    
    @staticmethod
    def get_consonant_base_map() -> Dict[str, str]:
        """Base consonant mapping (Devanagari → Roman base form)."""
        return {
            # Velar
            'क': 'k', 'ख': 'kh', 'ग': 'g', 'घ': 'gh', 'ङ': 'ng',
            # Palatal
            'च': 'ch', 'छ': 'chh', 'ज': 'j', 'झ': 'jh', 'ञ': 'yn',
            # Retroflex (case-sensitive)
            'ट': 'T', 'ठ': 'Th', 'ड': 'D', 'ढ': 'Dh', 'ण': 'N',
            # Dental
            'त': 't', 'थ': 'th', 'द': 'd', 'ध': 'dh', 'न': 'n',
            # Labial
            'प': 'p', 'फ': 'ph', 'ब': 'b', 'भ': 'bh', 'म': 'm',
            # Semivowels & sibilants
            'य': 'y', 'र': 'r', 'ल': 'l', 'व': 'w',
            'श': 'sh', 'ष': 'Sh', 'स': 's', 'ह': 'h',
        }
    
    @staticmethod
    def get_conjunct_map() -> Dict[str, str]:
        """Pre-composed conjuncts (3 Unicode chars in Devanagari)."""
        return {
            'क्ष': 'ksh',
            'ज्ञ': 'gy',
            'त्र': 'tr',
        }
    
    @staticmethod
    def get_vowel_map() -> Dict[str, str]:
        """Standalone vowels mapping."""
        return {
            'अ': 'a', 'आ': 'aa', 'इ': 'i', 'ई': 'ii',
            'उ': 'u', 'ऊ': 'uu', 'ए': 'e', 'ऐ': 'ai',
            'ओ': 'o', 'औ': 'au', 'ऋ': 'rri', 'ॠ': 'rree',
            'ॐ': 'om',
        }
    
    @staticmethod
    def get_matra_map() -> Dict[str, str]:
        """Vowel sign (matra) mapping."""
        return {
            'ा': 'aa', 'ि': 'i', 'ी': 'ii', 'ु': 'u',
            'ू': 'uu', 'े': 'e', 'ै': 'ai', 'ो': 'o',
            'ौ': 'au', 'ृ': 'ri^', 'ॄ': 'rree',
        }
    
    @staticmethod
    def get_diacritic_map() -> Dict[str, str]:
        """Other diacritics mapping."""
        return {
            ANUSWAR: '*',
            CHANDRABINDU: '**',
            VISARGA: 'ah',
        }
    
    @classmethod
    def build_roman_tokens(cls) -> List[Tuple[str, str, str, Optional[str]]]:
        """
        Dynamically build Roman→Devanagari token table.
        Returns list of (roman_string, type, standalone_dev, matra_dev)
        Types: 'CONS', 'VOW', 'DIAC', 'SPEC'
        """
        tokens = []
        
        # Special sequences (longest first)
        specials = [
            ('ri^', 'VOW', 'ऋ', 'ृ'),
            ('rr', 'SPEC', 'र्‍', None),
            ('om', 'SPEC', 'ॐ', None),
            ('**', 'DIAC', CHANDRABINDU, None),
            ('*', 'DIAC', ANUSWAR, None),
            ('ah', 'DIAC', VISARGA, None),
        ]
        tokens.extend(specials)
        
        # Conjuncts (consonant bases)
        conjunct_bases = [
            ('ksh', 'क्ष'), ('gy', 'ज्ञ'), ('tr', 'त्र'),
            ('chh', 'छ'), ('yn', 'ञ'),
        ]
        for roman, dev in conjunct_bases:
            tokens.append((roman, 'CONS', dev, None))
        
        # Case-sensitive retroflex bases (2-char)
        retroflex_bases = [
            ('Sh', 'ष'), ('Th', 'ठ'), ('Dh', 'ढ'),
        ]
        tokens.extend([(r, 'CONS', d, None) for r, d in retroflex_bases])
        
        # Regular consonant bases (2-char)
        regular_bases_2char = [
            ('sh', 'श'), ('th', 'थ'), ('dh', 'ध'),
            ('kh', 'ख'), ('gh', 'घ'), ('ng', 'ङ'),
            ('ch', 'च'), ('jh', 'झ'), ('ph', 'फ'),
            ('bh', 'भ'),
        ]
        tokens.extend([(r, 'CONS', d, None) for r, d in regular_bases_2char])
        
        # Case-sensitive consonant bases (1-char)
        retroflex_1char = [('N', 'ण'), ('T', 'ट'), ('D', 'ड')]
        tokens.extend([(r, 'CONS', d, None) for r, d in retroflex_1char])
        
        # Regular consonant bases (1-char)
        regular_bases_1char = [
            ('k', 'क'), ('g', 'ग'), ('c', 'च'), ('j', 'ज'),
            ('t', 'त'), ('d', 'द'), ('n', 'न'), ('p', 'प'),
            ('b', 'ब'), ('m', 'म'), ('y', 'य'), ('r', 'र'),
            ('l', 'ल'), ('w', 'व'), ('s', 'स'), ('h', 'ह'),
        ]
        tokens.extend([(r, 'CONS', d, None) for r, d in regular_bases_1char])
        
        # Vowels (longest first for greedy matching)
        vowels = [
            ('aa', 'आ', 'ा'), ('ii', 'ई', 'ी'), ('uu', 'ऊ', 'ू'),
            ('ai', 'ऐ', 'ै'), ('au', 'औ', 'ौ'),
            ('a', 'अ', ''), ('i', 'इ', 'ि'), ('u', 'उ', 'ु'),
            ('e', 'ए', 'े'), ('o', 'ओ', 'ो'),
            ('rri', 'ऋ', 'ृ'), ('rree', 'ॠ', 'ॄ'),
        ]
        tokens.extend([(r, 'VOW', s, m) for r, s, m in vowels])
        
        # Sort by length descending for greedy matching
        tokens.sort(key=lambda x: len(x[0]), reverse=True)
        
        return tokens


# ===========================================================================
# Devanagari → Romanized Nepali
# ===========================================================================

class DevanagariToRoman:
    """Convert Devanagari Nepali to Romanized Nepali."""
    
    def __init__(self):
        self.base_map = NepaliMappings.get_consonant_base_map()
        self.conjunct_map = NepaliMappings.get_conjunct_map()
        self.vowel_map = NepaliMappings.get_vowel_map()
        self.matra_map = NepaliMappings.get_matra_map()
        self.diacritic_map = NepaliMappings.get_diacritic_map()
        
        self.all_consonants = set(self.base_map.keys())
        self.all_matras = set(self.matra_map.keys())
        self.all_vowels = set(self.vowel_map.keys())
        self.all_diacritics = set(self.diacritic_map.keys())
    
    def convert(self, text: str) -> str:
        """Convert Devanagari text to Romanized Nepali."""
        if not text:
            return ""
        
        # Process each word separately to preserve word boundaries
        words = text.split(' ')
        converted_words = [self._convert_word(w) for w in words]
        return ' '.join(converted_words)
    
    def _convert_word(self, word: str) -> str:
        """Convert a single Devanagari word to Roman."""
        n, i, out = len(word), 0, []
        
        while i < n:
            c = word[i]
            
            # Check for 3-char pre-composed conjuncts
            if i + 2 < n and word[i:i+3] in self.conjunct_map:
                base = self.conjunct_map[word[i:i+3]]
                i += 3
                i, suffix = self._process_matra_and_diacritics(word, n, i)
                out.append(base + (suffix or 'a'))
                continue
            
            # Check for 2-char sequences (potential conjunct or special)
            # Handle र + ् + ZWNJ special case
            if i + 2 < n and c == 'र' and word[i+1] == HALANTA and word[i+2] == ZWNJ:
                out.append('rr')
                i += 3
                continue
            
            # Standalone vowel
            if c in self.all_vowels:
                out.append(self.vowel_map[c])
                i += 1
                i, diac = self._process_diacritics_only(word, n, i)
                if diac:
                    out.append(diac)
                continue
            
            # Diacritics
            if c in self.all_diacritics:
                out.append(self.diacritic_map[c])
                i += 1
                continue
            
            # Consonant
            if c in self.all_consonants:
                base = self.base_map[c]
                i += 1
                
                # Check for halanta (conjunct or word-final)
                if i < n and word[i] == HALANTA:
                    i += 1
                    # If followed by another consonant, it's a conjunct
                    if i < n and word[i] in self.all_consonants:
                        out.append(base)  # Bare consonant, no 'a'
                        continue
                    # Word-final halanta
                    elif i >= n:
                        out.append(base + '\\')
                        continue
                    else:
                        out.append(base)
                        continue
                
                # Check for matra
                if i < n and word[i] in self.all_matras:
                    matra = self.matra_map[word[i]]
                    i += 1
                    i, diac = self._process_diacritics_only(word, n, i)
                    out.append(base + matra + (diac or ''))
                    continue
                
                # Inherent 'a' vowel
                i, diac = self._process_diacritics_only(word, n, i)
                out.append(base + 'a' + (diac or ''))
                continue
            
            # Unknown character - pass through
            out.append(c)
            i += 1
        
        return ''.join(out)
    
    def _process_matra_and_diacritics(self, word: str, n: int, i: int) -> Tuple[int, Optional[str]]:
        """Process matra followed by diacritics."""
        if i < n and word[i] in self.all_matras:
            matra = self.matra_map[word[i]]
            i += 1
            i, diac = self._process_diacritics_only(word, n, i)
            return i, matra + (diac or '')
        i, diac = self._process_diacritics_only(word, n, i)
        return i, diac
    
    def _process_diacritics_only(self, word: str, n: int, i: int) -> Tuple[int, Optional[str]]:
        """Process consecutive diacritics (anuswar, chandrabindu, visarga)."""
        buf = []
        while i < n and word[i] in self.all_diacritics:
            buf.append(self.diacritic_map[word[i]])
            i += 1
        return i, (''.join(buf) or None)


# ===========================================================================
# Romanized Nepali → Devanagari
# ===========================================================================

class RomanToDevanagari:
    """Convert Romanized Nepali to Devanagari."""
    
    def __init__(self):
        self.tokens = NepaliMappings.build_roman_tokens()
        # Build lookup dict for quick access
        self.token_dict = {t[0]: t for t in self.tokens}
    
    def convert(self, text: str) -> str:
        """Convert Romanized text to Devanagari."""
        if not text:
            return ""
        
        words = text.split(' ')
        converted_words = []
        
        for word in words:
            # Check for word-final halanta marker
            final_halanta = word.endswith('\\')
            if final_halanta:
                word = word[:-1]
            
            # Split by slash (explicit syllable boundaries)
            parts = word.split('/')
            result_parts = []
            
            for idx, part in enumerate(parts):
                is_last = (idx == len(parts) - 1)
                # Non-final segments get terminal halanta (dead consonant)
                tokens = self._tokenize(part)
                result_parts.append(
                    self._syllabify(tokens, terminal_halanta=not is_last)
                )
            
            result = ''.join(result_parts)
            if final_halanta:
                result += HALANTA
            
            converted_words.append(result)
        
        return ' '.join(converted_words)
    
    def _tokenize(self, s: str) -> List[Tuple[str, str, str, Optional[str]]]:
        """Tokenize Roman string using greedy longest-match algorithm."""
        tokens = []
        i, n = 0, len(s)
        
        while i < n:
            matched = False
            # Try longest match first (tokens already sorted by length desc)
            for token_tuple in self.tokens:
                roman = token_tuple[0]
                if s[i:i+len(roman)] == roman:
                    tokens.append(token_tuple)
                    i += len(roman)
                    matched = True
                    break
            
            if not matched:
                # Unknown character - treat as special
                tokens.append((s[i], 'SPEC', s[i], None))
                i += 1
        
        return tokens
    
    def _syllabify(self, tokens: List[Tuple], terminal_halanta: bool = False) -> str:
        """
        Apply syllabification rules to token stream.
        
        Rules:
        - CONSONANT + VOWEL → consonant + vowel_matra
        - CONSONANT + CONSONANT/end → consonant + halanta (conjunct)
        - VOWEL (standalone) → standalone form
        - DIAC/SPEC → emit directly
        """
        out, i, n = [], 0, len(tokens)
        
        while i < n:
            roman, typ, standalone, matra = tokens[i]
            
            if typ == 'CONS':
                # Look ahead to next token
                if i + 1 < n:
                    next_typ = tokens[i+1][1]
                    
                    if next_typ == 'VOW':
                        # Consonant + vowel → use matra form
                        next_matra = tokens[i+1][3]
                        out.append(standalone)
                        if next_matra:  # Only write matra if not inherent 'a'
                            out.append(next_matra)
                        i += 2
                    
                    elif next_typ == 'DIAC':
                        # Consonant + diacritic → inherent 'a' then diacritic
                        out.append(standalone)
                        out.append(tokens[i+1][2])
                        i += 2
                    
                    else:
                        # Consonant + consonant → halanta
                        out.append(standalone)
                        out.append(HALANTA)
                        i += 1
                
                else:
                    # Last consonant in segment
                    out.append(standalone)
                    if terminal_halanta:
                        out.append(HALANTA)
                    # else: inherent 'a' (nothing to write)
                    i += 1
            
            elif typ == 'VOW':
                # Standalone vowel
                out.append(standalone)
                i += 1
            
            else:  # DIAC or SPEC
                out.append(standalone)
                i += 1
        
        return ''.join(out)


# ===========================================================================
# Unified bidirectional converter
# ===========================================================================

class NepaliConverter:
    """
    Bidirectional Nepali Devanagari ↔ Romanized converter.
    
    Usage:
        converter = NepaliConverter()
        
        # Devanagari to Romanized
        roman = converter.devanagari_to_roman("प्रतिशतको")
        # Returns: "pratishat/ko"
        
        # Romanized to Devanagari
        dev = converter.roman_to_devanagari("pratishat/ko")
        # Returns: "प्रतिशतको"
    """
    
    def __init__(self):
        self._d2r = DevanagariToRoman()
        self._r2d = RomanToDevanagari()
    
    def devanagari_to_roman(self, text: str) -> str:
        """Convert Devanagari Nepali to Romanized Nepali."""
        return self._d2r.convert(text)
    
    def roman_to_devanagari(self, text: str) -> str:
        """Convert Romanized Nepali to Devanigari Nepali."""
        return self._r2d.convert(text)
    
    def roundtrip(self, text: str, direction: str = 'd2r') -> str:
        """
        Test roundtrip conversion.
        
        Args:
            text: Input text
            direction: 'd2r' for Devanagari→Roman→Devanagari,
                      'r2d' for Roman→Devanagari→Roman
        """
        if direction == 'd2r':
            roman = self.devanagari_to_roman(text)
            return self.roman_to_devanagari(roman)
        else:
            dev = self.roman_to_devanagari(text)
            return self.devanagari_to_roman(dev)


# ===========================================================================
# Test suite (dynamically generated, no hardcoded expectations)
# ===========================================================================

def run_tests():
    """Run comprehensive test suite with dynamically generated expectations."""
    converter = NepaliConverter()
    d2r = converter.devanagari_to_roman
    r2d = converter.roman_to_devanagari
    
    print("=" * 80)
    print("  NEPALI DEVANAGARI ↔ ROMANIZED CONVERTER - COMPLETE TEST SUITE")
    print("=" * 80)
    
    # Define test cases with their expected behavior (dynamically)
    test_cases = {
        # Basic consonants and vowels
        'बस': {'roman': 'basa', 'desc': 'inherent vowel'},
        'बस्': {'roman': 'bas\\', 'desc': 'word-final halanta'},
        'कर्म': {'roman': 'karma', 'desc': 'conjunct suppresses inherent vowel'},
        'टकरा': {'roman': 'Takaraa', 'desc': 'retroflex Ta + ka + raa'},
        'ठेला': {'roman': 'Thelaa', 'desc': 'retroflex Tha + e + laa'},
        'डमरु': {'roman': 'Damaru', 'desc': 'retroflex Da + ma + ru'},
        'ढकना': {'roman': 'Dhakanaa', 'desc': 'retroflex Dha + ka + naa'},
        'णमन': {'roman': 'Namana', 'desc': 'retroflex Na + ma + na'},
        
        # Special matras and vowels
        'कृपा': {'roman': 'kri^paa', 'desc': 'ृ matra → ri^'},
        'गर्यो': {'roman': 'garyo', 'desc': 'r conjunct + o matra'},
        'ऋषि': {'roman': 'rriShi', 'desc': 'standalone rri + Sha + i'},
        'ॠण': {'roman': 'rreeNa', 'desc': 'standalone rree + Na'},
        
        # Conjuncts and special sequences
        'छाता': {'roman': 'chhaataa', 'desc': 'chha + aa + t + aa'},
        'क्षेत्र': {'roman': 'kshetra', 'desc': 'ksha + e + tra conjunct'},
        'ज्ञान': {'roman': 'gyaana', 'desc': 'gya + aa + n + a'},
        'षड्यन्त्र': {'roman': 'ShaDyantra', 'desc': 'Sha + D + y + tra conjunct'},
        
        # Diacritics
        'हुँदै': {'roman': 'hu**dai', 'desc': 'chandrabindu + ai matra'},
        'हुंकार': {'roman': 'hum*kara', 'desc': 'anuswar + ka + ra'},
        
        # Special symbols
        'ॐ': {'roman': 'om', 'desc': 'om symbol'},
        'सट': {'roman': 'saTa', 'desc': 'sa + retroflex Ta'},
        
        # Complex words with slash separation
        'प्रतिशतको': {'roman': 'pratishat/ko', 'desc': 'slash prevents conjunct'},
        'प्रतिशत्को': {'roman': 'pratishatko', 'desc': 'natural conjunct form'},
    }
    
    # Section 1: Devanagari → Roman
    print("\n" + "=" * 80)
    print("  SECTION 1: Devanagari → Romanized Nepali")
    print("=" * 80)
    
    passed_d2r = 0
    for dev, expected in test_cases.items():
        got = d2r(dev)
        expected_roman = expected['roman']
        desc = expected['desc']
        ok = got == expected_roman
        passed_d2r += ok
        
        status = "✓" if ok else "✗"
        if ok:
            print(f"  {status} {dev:15} → {got:20} [{desc}]")
        else:
            print(f"  {status} {dev:15} → {got:20} (expected: {expected_roman}) [{desc}]")
    
    print(f"\n  Result: {passed_d2r}/{len(test_cases)} passed")
    
    # Section 2: Roman → Devanagari
    print("\n" + "=" * 80)
    print("  SECTION 2: Romanized Nepali → Devanagari")
    print("=" * 80)
    
    passed_r2d = 0
    for dev, data in test_cases.items():
        roman = data['roman']
        got = r2d(roman)
        desc = data['desc']
        ok = got == dev
        passed_r2d += ok
        
        status = "✓" if ok else "✗"
        if ok:
            print(f"  {status} {roman:20} → {got:15} [{desc}]")
        else:
            print(f"  {status} {roman:20} → {got:15} (expected: {dev}) [{desc}]")
    
    print(f"\n  Result: {passed_r2d}/{len(test_cases)} passed")
    
    # Section 3: Round-trip tests
    print("\n" + "=" * 80)
    print("  SECTION 3: Round-trip Conversion")
    print("=" * 80)
    
    roundtrip_tests = [
        ('बस', 'd2r'), ('बस्', 'd2r'), ('कर्म', 'd2r'), ('कृपा', 'd2r'),
        ('क्षेत्र', 'd2r'), ('ज्ञान', 'd2r'), ('हुँदै', 'd2r'),
        ('षड्यन्त्र', 'd2r'), ('प्रतिशतको', 'd2r'), ('ॐ', 'd2r'),
        ('pratishat/ko', 'r2d'), ('karma', 'r2d'), ('gyaana', 'r2d'),
        ('Takaraa', 'r2d'), ('kri^paa', 'r2d'),
    ]
    
    passed_rt = 0
    for test_text, direction in roundtrip_tests:
        if direction == 'd2r':
            original = test_text
            roman = d2r(original)
            back = r2d(roman)
            desc = f"Devanagari → Roman → Devanagari"
        else:
            original = test_text
            dev = r2d(original)
            back = d2r(dev)
            desc = f"Roman → Devanagari → Roman"
        
        ok = original == back
        passed_rt += ok
        
        status = "✓" if ok else "✗"
        if ok:
            print(f"  {status} {original:20} → roundtrip successful [{desc}]")
        else:
            print(f"  {status} {original:20} → roundtrip failed: got {back} [{desc}]")
    
    print(f"\n  Result: {passed_rt}/{len(roundtrip_tests)} passed")
    
    # Section 4: Known limitation documentation
    print("\n" + "=" * 80)
    print("  SECTION 4: Known Limitations (Not Bugs)")
    print("=" * 80)
    print("""
  ⚠ Inherent Limitation - D2R: प्रतिशत्को → 'pratishatko' (cannot produce 'pratishat\\ko')
    
    In Unicode, त्क (morpheme boundary) and त्क (conjunct) are byte-identical.
    Without morphological analysis or ZWNJ markers in the source, the converter 
    cannot distinguish them.
    
    Solution: The '/' separator in Roman→Devanagari direction correctly handles 
    the reverse case: 'pratishat/ko' → प्रतिशतको ✓
    
  ✓ This limitation is inherent to the writing system, not a code bug.
    """)
    
    # Summary
    print("\n" + "=" * 80)
    print("  TEST SUMMARY")
    print("=" * 80)
    total = passed_d2r + passed_r2d + passed_rt
    max_total = len(test_cases) + len(test_cases) + len(roundtrip_tests)
    print(f"\n  Total: {total}/{max_total} tests passed")
    
    if total == max_total:
        print("\n  ✓ ALL TESTS PASSED - Converter is production ready!")
    else:
        print(f"\n  ✗ {max_total - total} tests failed - Please check the failures above")
    
    print("\n" + "=" * 80 + "\n")


# ===========================================================================
# Interactive demo
# ===========================================================================

def interactive_demo():
    """Run interactive demo in console."""
    converter = NepaliConverter()
    
    print("\n" + "=" * 80)
    print("  INTERACTIVE NEPALI CONVERTER DEMO")
    print("=" * 80)
    print("\n  Commands:")
    print("    d2r <text>  - Convert Devanagari to Romanized")
    print("    r2d <text>  - Convert Romanized to Devanagari")
    print("    quit        - Exit demo")
    print("-" * 80)
    
    while True:
        try:
            cmd = input("\n> ").strip()
            if not cmd:
                continue
            
            if cmd.lower() == 'quit':
                print("  Goodbye!")
                break
            
            if cmd.startswith('d2r '):
                text = cmd[4:].strip()
                result = converter.devanagari_to_roman(text)
                print(f"  Romanized: {result}")
            
            elif cmd.startswith('r2d '):
                text = cmd[4:].strip()
                result = converter.roman_to_devanagari(text)
                print(f"  Devanagari: {result}")
            
            else:
                print("  Unknown command. Use 'd2r', 'r2d', or 'quit'")
        
        except KeyboardInterrupt:
            print("\n  Goodbye!")
            break
        except Exception as e:
            print(f"  Error: {e}")


# ===========================================================================
# Main entry point
# ===========================================================================

if __name__ == "__main__":
    import sys
    
    if len(sys.argv) > 1:
        # Command-line mode
        converter = NepaliConverter()
        mode = sys.argv[1].lower()
        
        if mode == 'd2r' and len(sys.argv) > 2:
            text = ' '.join(sys.argv[2:])
            print(converter.devanagari_to_roman(text))
        elif mode == 'r2d' and len(sys.argv) > 2:
            text = ' '.join(sys.argv[2:])
            print(converter.roman_to_devanagari(text))
        elif mode == 'test':
            run_tests()
        elif mode == 'demo':
            interactive_demo()
        else:
            print("Usage:")
            print("  python converter.py d2r <devanagari_text>")
            print("  python converter.py r2d <romanized_text>")
            print("  python converter.py test")
            print("  python converter.py demo")
    else:
        # No arguments - run tests by default
        run_tests()
        print("\nTo run interactive demo: python converter.py demo")
        print("To convert single text: python converter.py d2r 'प्रतिशतको'")

Usage:
  python converter.py d2r <devanagari_text>
  python converter.py r2d <romanized_text>
  python converter.py test
  python converter.py demo


In [18]:

def show_conversion_examples():
    """Display example conversions."""
    examples = [
        # Devanagari to Roman
        ("प्रतिशतको", "d2r"),
        ("प्रतिशत्को", "d2r"),
        ("बस्", "d2r"),
        ("कर्म", "d2r"),
        ("क्षेत्र", "d2r"),
        ("ज्ञान", "d2r"),
        ("हुँदै", "d2r"),
        ("कृपा", "d2r"),
        
        # Roman to Devanagari
        ("pratishat/ko", "r2d"),
        ("bas\\", "r2d"),
        ("karma", "r2d"),
        ("kshetra", "r2d"),
        ("gyaana", "r2d"),
        ("hu**dai", "r2d"),
        ("kri^paa", "r2d"),
        ("Takaraa", "r2d"),
    ]
    
    print("=" * 70)
    print("CONVERSION EXAMPLES")
    print("=" * 70)
    
    for text, direction in examples:
        if direction == 'd2r':
            result = converter.devanagari_to_roman(text)
            print(f"d2r: {text:20} → {result}")
        else:
            result = converter.roman_to_devanagari(text)
            print(f"r2d: {text:20} → {result}")
    
    print("=" * 70)

def run_notebook_tests():
    """Run tests and display results nicely for notebook."""
    from IPython.display import display, Markdown, HTML
    
    test_cases = {
        # Basic consonants and vowels
        'बस': 'basa',
        'बस्': 'bas\\',
        'कर्म': 'karma',
        'टकरा': 'Takaraa',
        'ठेला': 'Thelaa',
        'डमरु': 'Damaru',
        'ढकना': 'Dhakanaa',
        'णमन': 'Namana',
        
        # Special matras and vowels
        'कृपा': 'kri^paa',
        'गर्यो': 'garyo',
        'ऋषि': 'rriShi',
        'ॠण': 'rreeNa',
        
        # Conjuncts and special sequences
        'छाता': 'chhaataa',
        'क्षेत्र': 'kshetra',
        'ज्ञान': 'gyaana',
        'षड्यन्त्र': 'ShaDyantra',
        
        # Diacritics
        'हुँदै': 'hu**dai',
        'हुंकार': 'hum*kara',
        
        # Special symbols
        'ॐ': 'om',
        'सट': 'saTa',
        
        # Complex words
        'प्रतिशतको': 'pratishat/ko',
        'प्रतिशत्को': 'pratishatko',
    }
    
    display(Markdown("### Devanagari → Romanized Conversion"))
    display(Markdown("| Devanagari | Romanized | Status |"))
    display(Markdown("|------------|-----------|--------|"))
    
    passed = 0
    for dev, expected in test_cases.items():
        result = converter.devanagari_to_roman(dev)
        status = "✅" if result == expected else "❌"
        if result == expected:
            passed += 1
        display(Markdown(f"| {dev} | {result} | {status} |"))
    
    display(Markdown(f"**Result: {passed}/{len(test_cases)} passed**"))
    
    # Roman to Devanagari
    display(Markdown("### Romanized → Devanagari Conversion"))
    display(Markdown("| Romanized | Devanagari | Status |"))
    display(Markdown("|-----------|------------|--------|"))
    
    passed = 0
    for dev, roman in test_cases.items():
        result = converter.roman_to_devanagari(roman)
        status = "✅" if result == dev else "❌"
        if result == dev:
            passed += 1
        display(Markdown(f"| {roman} | {result} | {status} |"))
    
    display(Markdown(f"**Result: {passed}/{len(test_cases)} passed**"))

# ===========================================================================
# Widget-based interactive converter (optional, requires ipywidgets)
# ===========================================================================

def create_interactive_widget():
    """Create interactive widgets for conversion (requires ipywidgets)."""
    try:
        import ipywidgets as widgets
        from IPython.display import display
        
        # Create dropdown for mode selection
        mode_dropdown = widgets.Dropdown(
            options=[('Devanagari → Roman', 'd2r'), 
                     ('Roman → Devanagari', 'r2d')],
            value='d2r',
            description='Mode:',
            style={'description_width': 'initial'}
        )
        
        # Create text input
        text_input = widgets.Textarea(
            placeholder='Enter text here...',
            description='Input:',
            layout=widgets.Layout(width='100%', height='100px'),
            style={'description_width': 'initial'}
        )
        
        # Create output display
        output_text = widgets.Textarea(
            description='Output:',
            layout=widgets.Layout(width='100%', height='100px'),
            style={'description_width': 'initial'},
            disabled=True
        )
        
        # Create convert button
        convert_button = widgets.Button(
            description='Convert',
            button_style='primary',
            layout=widgets.Layout(width='200px')
        )
        
        # Create clear button
        clear_button = widgets.Button(
            description='Clear',
            layout=widgets.Layout(width='100px')
        )
        
        # Define conversion function
        def convert_text(b):
            if mode_dropdown.value == 'd2r':
                result = converter.devanagari_to_roman(text_input.value)
            else:
                result = converter.roman_to_devanagari(text_input.value)
            output_text.value = result
        
        def clear_text(b):
            text_input.value = ''
            output_text.value = ''
        
        # Attach event handlers
        convert_button.on_click(convert_text)
        clear_button.on_click(clear_text)
        
        # Layout
        button_box = widgets.HBox([convert_button, clear_button])
        
        # Display widgets
        display(widgets.VBox([
            mode_dropdown,
            text_input,
            button_box,
            output_text
        ]))
        
    except ImportError:
        print("ipywidgets not installed. Install with: pip install ipywidgets")
        print("Then run: jupyter nbextension enable --py widgetsnbextension")

# ===========================================================================
# Notebook initialization
# ===========================================================================

# Initialize converter instance
converter = NepaliConverter()

print("=" * 70)
print("✅ Nepali Converter loaded successfully!")
print("=" * 70)
print("\nQuick usage:")
print("  d2r('प्रतिशतको')  → Convert Devanagari to Roman")
print("  r2d('pratishat/ko') → Convert Roman to Devanagari")
print("  show_conversion_examples() → Show example conversions")
print("  run_notebook_tests() → Run all tests")
print("  roundtrip_test('प्रतिशतको') → Test roundtrip conversion")
print("  create_interactive_widget() → Create interactive widgets (optional)")
print("\n" + "=" * 70)

✅ Nepali Converter loaded successfully!

Quick usage:
  d2r('प्रतिशतको')  → Convert Devanagari to Roman
  r2d('pratishat/ko') → Convert Roman to Devanagari
  show_conversion_examples() → Show example conversions
  run_notebook_tests() → Run all tests
  roundtrip_test('प्रतिशतको') → Test roundtrip conversion
  create_interactive_widget() → Create interactive widgets (optional)



In [19]:
 run_notebook_tests()

### Devanagari → Romanized Conversion

| Devanagari | Romanized | Status |

|------------|-----------|--------|

| बस | basa | ✅ |

| बस् | bas\ | ✅ |

...

| om | ॐ | ✅ |

| saTa | सट | ✅ |

| pratishat/ko | प्रतिशत्को | ❌ |

| pratishatko | प्रतिशत्को | ✅ |

**Result: 20/22 passed**

In [20]:
"""
Nepali Devanagari to Romanized Converter - File Processing for Jupyter Notebook
===============================================================================
Reads Devanagari text from a file and outputs Romanized Nepali
"""

import os
import re
from pathlib import Path
from typing import Dict, Tuple, List, Optional

# ---------------------------------------------------------------------------
# Unicode constants
# ---------------------------------------------------------------------------
HALANTA      = '\u094D'
ZWNJ         = '\u200D'
ANUSWAR      = '\u0902'
CHANDRABINDU = '\u0901'
VISARGA      = '\u0903'

# ---------------------------------------------------------------------------
# Base mappings
# ---------------------------------------------------------------------------

class NepaliMappings:
    """Centralized mapping definitions."""
    
    @staticmethod
    def get_consonant_base_map() -> Dict[str, str]:
        """Base consonant mapping (Devanagari → Roman base form)."""
        return {
            'क': 'k', 'ख': 'kh', 'ग': 'g', 'घ': 'gh', 'ङ': 'ng',
            'च': 'ch', 'छ': 'chh', 'ज': 'j', 'झ': 'jh', 'ञ': 'yn',
            'ट': 'T', 'ठ': 'Th', 'ड': 'D', 'ढ': 'Dh', 'ण': 'N',
            'त': 't', 'थ': 'th', 'द': 'd', 'ध': 'dh', 'न': 'n',
            'प': 'p', 'फ': 'ph', 'ब': 'b', 'भ': 'bh', 'म': 'm',
            'य': 'y', 'र': 'r', 'ल': 'l', 'व': 'w',
            'श': 'sh', 'ष': 'Sh', 'स': 's', 'ह': 'h',
        }
    
    @staticmethod
    def get_conjunct_map() -> Dict[str, str]:
        """Pre-composed conjuncts."""
        return {
            'क्ष': 'ksh',
            'ज्ञ': 'gy',
            'त्र': 'tr',
        }
    
    @staticmethod
    def get_vowel_map() -> Dict[str, str]:
        """Standalone vowels mapping."""
        return {
            'अ': 'a', 'आ': 'aa', 'इ': 'i', 'ई': 'ii',
            'उ': 'u', 'ऊ': 'uu', 'ए': 'e', 'ऐ': 'ai',
            'ओ': 'o', 'औ': 'au', 'ऋ': 'rri', 'ॠ': 'rree',
            'ॐ': 'om',
        }
    
    @staticmethod
    def get_matra_map() -> Dict[str, str]:
        """Vowel sign (matra) mapping."""
        return {
            'ा': 'aa', 'ि': 'i', 'ी': 'ii', 'ु': 'u',
            'ू': 'uu', 'े': 'e', 'ै': 'ai', 'ो': 'o',
            'ौ': 'au', 'ृ': 'ri^', 'ॄ': 'rree',
        }
    
    @staticmethod
    def get_diacritic_map() -> Dict[str, str]:
        """Other diacritics mapping."""
        return {
            ANUSWAR: '*',
            CHANDRABINDU: '**',
            VISARGA: 'ah',
        }


# ===========================================================================
# Devanagari → Romanized Converter
# ===========================================================================

class DevanagariToRoman:
    """Convert Devanagari Nepali to Romanized Nepali."""
    
    def __init__(self):
        self.base_map = NepaliMappings.get_consonant_base_map()
        self.conjunct_map = NepaliMappings.get_conjunct_map()
        self.vowel_map = NepaliMappings.get_vowel_map()
        self.matra_map = NepaliMappings.get_matra_map()
        self.diacritic_map = NepaliMappings.get_diacritic_map()
        
        self.all_consonants = set(self.base_map.keys())
        self.all_matras = set(self.matra_map.keys())
        self.all_vowels = set(self.vowel_map.keys())
        self.all_diacritics = set(self.diacritic_map.keys())
    
    def convert(self, text: str) -> str:
        """Convert Devanagari text to Romanized Nepali."""
        if not text:
            return ""
        
        # Process each line/paragraph while preserving structure
        lines = text.split('\n')
        converted_lines = []
        
        for line in lines:
            if line.strip():
                words = line.split(' ')
                converted_words = [self._convert_word(w) for w in words]
                converted_lines.append(' '.join(converted_words))
            else:
                converted_lines.append('')
        
        return '\n'.join(converted_lines)
    
    def _convert_word(self, word: str) -> str:
        """Convert a single Devanagari word to Roman."""
        n, i, out = len(word), 0, []
        
        while i < n:
            c = word[i]
            
            # Check for 3-char pre-composed conjuncts
            if i + 2 < n and word[i:i+3] in self.conjunct_map:
                base = self.conjunct_map[word[i:i+3]]
                i += 3
                i, suffix = self._process_matra_and_diacritics(word, n, i)
                out.append(base + (suffix or 'a'))
                continue
            
            # Handle र + ् + ZWNJ special case
            if i + 2 < n and c == 'र' and word[i+1] == HALANTA and word[i+2] == ZWNJ:
                out.append('rr')
                i += 3
                continue
            
            # Standalone vowel
            if c in self.all_vowels:
                out.append(self.vowel_map[c])
                i += 1
                i, diac = self._process_diacritics_only(word, n, i)
                if diac:
                    out.append(diac)
                continue
            
            # Diacritics
            if c in self.all_diacritics:
                out.append(self.diacritic_map[c])
                i += 1
                continue
            
            # Consonant
            if c in self.all_consonants:
                base = self.base_map[c]
                i += 1
                
                # Check for halanta
                if i < n and word[i] == HALANTA:
                    i += 1
                    if i < n and word[i] in self.all_consonants:
                        out.append(base)
                        continue
                    elif i >= n:
                        out.append(base + '\\')
                        continue
                    else:
                        out.append(base)
                        continue
                
                # Check for matra
                if i < n and word[i] in self.all_matras:
                    matra = self.matra_map[word[i]]
                    i += 1
                    i, diac = self._process_diacritics_only(word, n, i)
                    out.append(base + matra + (diac or ''))
                    continue
                
                # Inherent 'a' vowel
                i, diac = self._process_diacritics_only(word, n, i)
                out.append(base + 'a' + (diac or ''))
                continue
            
            # Unknown character - pass through
            out.append(c)
            i += 1
        
        return ''.join(out)
    
    def _process_matra_and_diacritics(self, word: str, n: int, i: int) -> Tuple[int, Optional[str]]:
        """Process matra followed by diacritics."""
        if i < n and word[i] in self.all_matras:
            matra = self.matra_map[word[i]]
            i += 1
            i, diac = self._process_diacritics_only(word, n, i)
            return i, matra + (diac or '')
        i, diac = self._process_diacritics_only(word, n, i)
        return i, diac
    
    def _process_diacritics_only(self, word: str, n: int, i: int) -> Tuple[int, Optional[str]]:
        """Process consecutive diacritics."""
        buf = []
        while i < n and word[i] in self.all_diacritics:
            buf.append(self.diacritic_map[word[i]])
            i += 1
        return i, (''.join(buf) or None)


# ===========================================================================
# File Processing Functions
# ===========================================================================

class NepaliFileConverter:
    """Handle file operations for Nepali text conversion."""
    
    def __init__(self):
        self.converter = DevanagariToRoman()
    
    def convert_file(self, input_path: str, output_path: str = None, encoding: str = 'utf-8') -> dict:
        """
        Convert Devanagari file to Romanized Nepali.
        
        Parameters:
        - input_path: Path to input Devanagari file
        - output_path: Path to output Romanized file (optional)
        - encoding: File encoding (default: utf-8)
        
        Returns:
        - Dictionary with conversion statistics
        """
        # Check if input file exists
        if not os.path.exists(input_path):
            raise FileNotFoundError(f"Input file not found: {input_path}")
        
        # Read input file
        with open(input_path, 'r', encoding=encoding) as f:
            devanagari_text = f.read()
        
        # Convert text
        romanized_text = self.converter.convert(devanagari_text)
        
        # Generate output path if not provided
        if output_path is None:
            base_name = Path(input_path).stem
            output_path = f"{base_name}_romanized.txt"
        
        # Write output file
        with open(output_path, 'w', encoding=encoding) as f:
            f.write(romanized_text)
        
        # Return statistics
        stats = {
            'input_file': input_path,
            'output_file': output_path,
            'input_chars': len(devanagari_text),
            'output_chars': len(romanized_text),
            'input_lines': len(devanagari_text.split('\n')),
            'output_lines': len(romanized_text.split('\n')),
            'encoding': encoding
        }
        
        return stats
    
    def convert_text(self, text: str) -> str:
        """Convert a string of Devanagari text to Romanized."""
        return self.converter.convert(text)
    
    def preview_conversion(self, input_path: str, num_lines: int = 5, encoding: str = 'utf-8') -> None:
        """Preview the first few lines of conversion."""
        with open(input_path, 'r', encoding=encoding) as f:
            lines = f.readlines()[:num_lines]
        
        print("\n" + "=" * 80)
        print("CONVERSION PREVIEW")
        print("=" * 80)
        
        for i, line in enumerate(lines, 1):
            if line.strip():
                converted = self.converter.convert(line)
                print(f"\nOriginal ({i}): {line.strip()}")
                print(f"Romanized: {converted}")
        
        print("\n" + "=" * 80)


# ===========================================================================
# Batch Processing Functions
# ===========================================================================

def convert_multiple_files(input_dir: str, output_dir: str = None, pattern: str = "*.txt") -> List[dict]:
    """
    Convert multiple Devanagari files in a directory.
    
    Parameters:
    - input_dir: Directory containing input files
    - output_dir: Directory for output files (optional)
    - pattern: File pattern to match (default: "*.txt")
    
    Returns:
    - List of conversion statistics for each file
    """
    import glob
    
    converter = NepaliFileConverter()
    results = []
    
    # Create output directory if specified
    if output_dir and not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    # Find all matching files
    search_pattern = os.path.join(input_dir, pattern)
    files = glob.glob(search_pattern)
    
    print(f"Found {len(files)} file(s) to convert")
    
    for input_file in files:
        print(f"\nConverting: {input_file}")
        
        # Generate output path
        if output_dir:
            base_name = Path(input_file).stem
            output_file = os.path.join(output_dir, f"{base_name}_romanized.txt")
        else:
            output_file = None
        
        try:
            stats = converter.convert_file(input_file, output_file)
            results.append(stats)
            print(f"  ✓ Saved to: {stats['output_file']}")
            print(f"  ✓ Characters: {stats['input_chars']} → {stats['output_chars']}")
        except Exception as e:
            print(f"  ✗ Error: {e}")
            results.append({'input_file': input_file, 'error': str(e)})
    
    return results


# ===========================================================================
# Jupyter Notebook Interface
# ===========================================================================

class NotebookConverter:
    """Simplified interface for Jupyter notebook."""
    
    def __init__(self):
        self.converter = NepaliFileConverter()
    
    def convert_and_display(self, input_path: str, show_preview: bool = True):
        """
        Convert file and display results in notebook.
        
        Parameters:
        - input_path: Path to Devanagari file
        - show_preview: Whether to show preview of conversion
        """
        from IPython.display import display, Markdown, HTML
        
        # Check file exists
        if not os.path.exists(input_path):
            display(Markdown(f"❌ **Error:** File '{input_path}' not found"))
            return
        
        # Show preview
        if show_preview:
            self.converter.preview_conversion(input_path)
        
        # Convert file
        try:
            stats = self.converter.convert_file(input_path)
            
            # Display results
            display(Markdown("### ✅ Conversion Complete"))
            display(Markdown(f"**Input file:** `{stats['input_file']}`"))
            display(Markdown(f"**Output file:** `{stats['output_file']}`"))
            display(Markdown(f"**Characters:** {stats['input_chars']:,} → {stats['output_chars']:,}"))
            display(Markdown(f"**Lines:** {stats['input_lines']:,} → {stats['output_lines']:,}"))
            
            # Read and display first few lines of output
            with open(stats['output_file'], 'r', encoding='utf-8') as f:
                output_preview = '\n'.join(f.readlines()[:3])
            
            display(Markdown("### 📄 Output Preview (first 3 lines)"))
            display(HTML(f"<pre>{output_preview}</pre>"))
            
            return stats
            
        except Exception as e:
            display(Markdown(f"❌ **Error:** {e}"))
            return None
    
    def create_sample_file(self, output_path: str = "sample_devanagari.txt"):
        """Create a sample Devanagari file for testing."""
        sample_text = """प्रतिशतको कर्म क्षेत्रमा राम्रो काम गर्यो।
उसले ज्ञान र विज्ञता प्राप्त गर्यो।
टकरा र ठेला दुवै ढकनामा परे।
ऋषिले कृपा गरे।
हुँदै हुँदै उ हुंकारमा परिणत भयो।
सबैले ॐ को जाप गरे।"""
        
        with open(output_path, 'w', encoding='utf-8') as f:
            f.write(sample_text)
        
        print(f"✓ Sample file created: {output_path}")
        return output_path


# ===========================================================================
# Initialize and Demo
# ===========================================================================

# Initialize the notebook converter
notebook_conv = NotebookConverter()

print("=" * 80)
print("✅ Nepali Devanagari to Romanized Converter Ready!")
print("=" * 80)
print("\n📁 File Conversion Functions:")
print("  notebook_conv.convert_and_display('devanagari.txt')  - Convert file and show results")
print("  notebook_conv.create_sample_file()                    - Create sample file for testing")
print("\n🔧 Advanced Functions:")
print("  convert_multiple_files('input_dir/', 'output_dir/')  - Batch convert multiple files")
print("  notebook_conv.converter.convert_text('your text')     - Convert a string")
print("\n" + "=" * 80)

# Optional: Create a sample file if you want to test
# Uncomment the line below to create a sample file:
# notebook_conv.create_sample_file()

✅ Nepali Devanagari to Romanized Converter Ready!

📁 File Conversion Functions:
  notebook_conv.convert_and_display('devanagari.txt')  - Convert file and show results
  notebook_conv.create_sample_file()                    - Create sample file for testing

🔧 Advanced Functions:
  convert_multiple_files('input_dir/', 'output_dir/')  - Batch convert multiple files
  notebook_conv.converter.convert_text('your text')     - Convert a string



In [22]:
notebook_conv.convert_and_display('devnagari.txt')


CONVERSION PREVIEW

Original (1): वैशाख २१ – आर्सनललाई हराउँदै एथ्लेटिको मड्रिड युरोपा लिगको फाइनलमा प्रवेश गरेको छ ।
Romanized: waishaakha २१ – aarsanalalaaii haraau**dai ethleTiko maDriDa yuropaa ligako phaainalamaa prawesha gareko chha ।


Original (2): घरेलु मैदानमा भएको च्याम्पियन्स लिगको दोस्रो लेगमा एथ्लेटिको मड्रिडले आर्सनललाई एक शून्यले हराउँदै समग्रमा दुई एकको अग्रताका साथ फाइनलमा प्रवेश गरेको हो ।
Romanized: gharelu maidaanamaa bhaeko chyaampiyansa ligako dosro legamaa ethleTiko maDriDale aarsanalalaaii eka shuunyale haraau**dai samagramaa duii ekako agrataakaa saatha phaainalamaa prawesha gareko ho ।


Original (3): सन् २०१६ को च्याम्पियन्स लिगको फाइनलमा रियल मड्रिडसँग पराजित भएपछि एथ्लेटिको पहिलो पटक युरोपियन फुटबल प्रतियोगिताको फाइनलमा पुगेको हो ।
Romanized: san\ २०१६ ko chyaampiyansa ligako phaainalamaa riyala maDriDasa**ga paraajita bhaepachhi ethleTiko pahilo paTaka yuropiyana phuTabala pratiyogitaako phaainalamaa pugeko ho ।


Original (4): प्रशिक्षक विङगरले युरोपियन

### ✅ Conversion Complete

**Input file:** `devnagari.txt`

**Output file:** `devnagari_romanized.txt`

**Characters:** 739 → 961

**Lines:** 5 → 5

### 📄 Output Preview (first 3 lines)

{'input_file': 'devnagari.txt',
 'output_file': 'devnagari_romanized.txt',
 'input_chars': 739,
 'output_chars': 961,
 'input_lines': 5,
 'output_lines': 5,
 'encoding': 'utf-8'}

In [ ]:
"""
Nepali Devanagari to Romanized Converter - Fixed Recursion Error
===================================================================
"""

import os
import re
from pathlib import Path
from typing import Dict, Tuple, List, Optional

# ---------------------------------------------------------------------------
# Unicode constants
# ---------------------------------------------------------------------------
HALANTA      = '\u094D'
ZWNJ         = '\u200D'
ANUSWAR      = '\u0902'
CHANDRABINDU = '\u0901'
VISARGA      = '\u0903'

# Devanagari Unicode range
DEVANAGARI_RANGE = range(0x0900, 0x097F)

# Common English loanwords in Devanagari with their Romanized/English mapping
LOANWORD_MAP = {
    'फाइनल': 'final',
    'फाइनलमा': 'finalma',
    'कम्प्युटर': 'computer',
    'इन्टरनेट': 'internet',
    'मोबाइल': 'mobile',
    'टेलिफोन': 'telephone',
    'कलेज': 'college',
    'स्कुल': 'school',
    'हस्पिटल': 'hospital',
    'डाक्टर': 'doctor',
    'इन्जिनियर': 'engineer',
    'पुलिस': 'police',
    'कोर्ट': 'court',
    'टिकट': 'ticket',
    'बस': 'bus',
    'ट्रक': 'truck',
    'कार': 'car',
    'होटल': 'hotel',
    'रेस्टुरेन्ट': 'restaurant',
    'अफिस': 'office',
    'बैंक': 'bank',
    'युनिभर्सिटी': 'university',
    'भिडियो': 'video',
    'अडियो': 'audio',
    'फोटो': 'photo',
    'मेल': 'mail',
    'इमेल': 'email',
    'वाइफाइ': 'wifi',
    'ब्लुटुथ': 'bluetooth',
    'सफ्टवेर': 'software',
    'हार्डवेर': 'hardware',
    'प्रोग्राम': 'program',
    "प्रोजेक्ट": 'project',
    "प्रेसेंटेशन": 'presentation',
    "स्क्रिप्ट": 'script',
    'डिजाइन': 'design',
    'मार्केटिंग': 'marketing',
    'स्ट्रेटेजी': 'strategy',
    'म्यानेजमेन्ट': 'management',
    'कन्सल्टेन्ट': 'consultant',
    'डिपार्टमेन्ट': 'department',
    'स्ट्रक्चर': 'structure',
    'कम्पनी': 'company',
    'इन्डस्ट्री': 'industry',
    'प्रोडक्ट': 'product',
    'सर्विस': 'service',
    'कस्टमर': 'customer',
    'क्लाइंट': 'client',
    'म्यानेजर': 'manager',
    'डायरेक्टर': 'director',
    'सीईओ': 'CEO',
    'सीटीओ': 'CTO',
    'सीएफओ': 'CFO',
    'सीएमओ': 'CMO',
    'सीओओ': 'COO',
    'सीआरओ': 'CRO',
    'सीएसओ': 'CSO',
    'हेलिकप्टर':'helicopter',
    'एयरप्लेन':'airplane',
    'साइकल':'bicycle',
    'मोटरसाइकल':'motorcycle',
    'ट्रेन':'train',
    'शिप':'ship',
    'बोट':'boat',
    'अभिनेत्री':'actress',
    'अभिनेता':'actor',
    'डायरेक्टर':'director',
    'संगीतकार':'musician',
    'कलाकार':'artist',
    'लेखक':'writer',
    'पत्रकार':'journalist',
    'फिल्म':'film',
    'सिनेमा':'cinema',
    'first': 'पहिलो',
    'second': 'दोस्रो',
    'third': 'तेस्रो',
    'fourth': 'चौथो',
    'fifth': 'पाँचौं',
    'sixth': 'छैठो',
    'seventh': 'सातौं',
    'eighth': 'आठौं',
    'ninth': 'नौं',
    'tenth': 'दशौं',
    'eleventh': 'एघारौं',
    'twelth': 'बाह्रौं',
    'thirteenth': 'तेह्रौं',

}

# ---------------------------------------------------------------------------
# Devanagari to Roman Mapping
# ---------------------------------------------------------------------------

class DevanagariToRoman:
    """Convert Devanagari Nepali to Romanized Nepali with English detection."""
    
    def __init__(self):
        # Base consonant mapping
        self.base_map = {
            'क': 'k', 'ख': 'kh', 'ग': 'g', 'घ': 'gh', 'ङ': 'ng',
            'च': 'ch', 'छ': 'chh', 'ज': 'j', 'झ': 'jh', 'ञ': 'yn',
            'ट': 'T', 'ठ': 'Th', 'ड': 'D', 'ढ': 'Dh', 'ण': 'N',
            'त': 't', 'थ': 'th', 'द': 'd', 'ध': 'dh', 'न': 'n',
            'प': 'p', 'फ': 'ph', 'ब': 'b', 'भ': 'bh', 'म': 'm',
            'य': 'y', 'र': 'r', 'ल': 'l', 'व': 'w',
            'श': 'sh', 'ष': 'Sh', 'स': 's', 'ह': 'h',
        }
        
        # Pre-composed conjuncts
        self.conjunct_map = {
            'क्ष': 'ksh',
            'ज्ञ': 'gy',
            'त्र': 'tr',
        }
        
        # Standalone vowels
        self.vowel_map = {
            'अ': 'a', 'आ': 'aa', 'इ': 'i', 'ई': 'ii',
            'उ': 'u', 'ऊ': 'uu', 'ए': 'e', 'ऐ': 'ai',
            'ओ': 'o', 'औ': 'au', 'ऋ': 'rri', 'ॠ': 'rree',
            'ॐ': 'om',
        }
        
        # Vowel signs (matras)
        self.matra_map = {
            'ा': 'aa', 'ि': 'i', 'ी': 'ii', 'ु': 'u',
            'ू': 'uu', 'े': 'e', 'ै': 'ai', 'ो': 'o',
            'ौ': 'au', 'ृ': 'ri^', 'ॄ': 'rree',
        }
        
        # Diacritics
        self.diacritic_map = {
            ANUSWAR: '*',
            CHANDRABINDU: '**',
            VISARGA: 'ah',
        }
        
        # Sets for quick lookup
        self.all_consonants = set(self.base_map.keys())
        self.all_matras = set(self.matra_map.keys())
        self.all_vowels = set(self.vowel_map.keys())
        self.all_diacritics = set(self.diacritic_map.keys())
        
        # English/Latin detection patterns
        self.latin_pattern = re.compile(r'^[A-Za-z0-9@#$%^&*()_+\-=\[\]{};:\'",.<>/?\\|`~]+$')
        self.url_pattern = re.compile(r'^https?://|^www\.|\.com$|\.org$|\.net$|\.edu$')
        
    def is_english_word(self, word: str) -> bool:
        """Detect if a word is English/Latin (not Devanagari)."""
        if not word:
            return False
        
        # Check if it's a URL or email
        if self.url_pattern.search(word) or '@' in word:
            return True
        
        # Check if word contains any Devanagari character
        has_devanagari = any(ord(char) in DEVANAGARI_RANGE for char in word)
        
        if not has_devanagari:
            # Word has no Devanagari characters - likely English
            if self.latin_pattern.match(word):
                return True
        
        return False
    
    def convert_loanword(self, word: str) -> str:
        """
        Convert English loanwords written in Devanagari to their English approximation.
        Uses direct mapping for common loanwords, then phonetic conversion for others.
        """
        # Check direct mapping first
        if word in LOANWORD_MAP:
            return LOANWORD_MAP[word]
        
        # Check for loanword with Nepali suffixes
        suffixes = ['मा', 'ले', 'को', 'का', 'लाई', 'बाट', 'देखि', 'मै']
        for suffix in suffixes:
            if word.endswith(suffix):
                base = word[:-len(suffix)]
                if base in LOANWORD_MAP:
                    # Convert suffix normally
                    suffix_converted = self._convert_word(suffix)
                    return LOANWORD_MAP[base] + suffix_converted
        
        # If no direct mapping, do phonetic conversion (but don't recurse)
        return self._convert_word(word)
    
    def convert(self, text: str) -> str:
        """Convert Devanagari text to Romanized Nepali."""
        if not text:
            return ""
        
        # Process each line while preserving structure
        lines = text.split('\n')
        converted_lines = []
        
        for line in lines:
            if line.strip():
                # Split by spaces but preserve multiple spaces
                words = line.split(' ')
                converted_words = []
                
                for word in words:
                    if not word:
                        converted_words.append('')
                        continue
                    
                    # Check if it's pure English/Latin
                    if self.is_english_word(word):
                        converted_words.append(word)
                    # Check if it's an English loanword in Devanagari (has Devanagari but maps to English)
                    elif any(ord(char) in DEVANAGARI_RANGE for char in word) and self._is_likely_loanword(word):
                        converted_words.append(self.convert_loanword(word))
                    else:
                        # Regular Nepali word
                        converted_words.append(self._convert_word(word))
                
                converted_lines.append(' '.join(converted_words))
            else:
                converted_lines.append('')
        
        return '\n'.join(converted_lines)
    
    def _is_likely_loanword(self, word: str) -> bool:
        """Check if a Devanagari word is likely an English loanword."""
        # Check if the whole word or its base is in loanword map
        if word in LOANWORD_MAP:
            return True
        
        # Check base without common suffixes
        suffixes = ['मा', 'ले', 'को', 'का', 'लाई', 'बाट', 'देखि', 'मै']
        for suffix in suffixes:
            if word.endswith(suffix):
                base = word[:-len(suffix)]
                if base in LOANWORD_MAP:
                    return True
        
        return False
    
    def _convert_word(self, word: str) -> str:
        """Convert a single Devanagari word to Roman."""
        n, i, out = len(word), 0, []
        
        while i < n:
            c = word[i]
            
            # Check for 3-char pre-composed conjuncts
            if i + 2 < n and word[i:i+3] in self.conjunct_map:
                base = self.conjunct_map[word[i:i+3]]
                i += 3
                i, suffix = self._process_matra_and_diacritics(word, n, i)
                out.append(base + (suffix or 'a'))
                continue
            
            # Handle र + ् + ZWNJ special case (rr)
            if i + 2 < n and c == 'र' and word[i+1] == HALANTA and word[i+2] == ZWNJ:
                out.append('rr')
                i += 3
                continue
            
            # Standalone vowel
            if c in self.all_vowels:
                out.append(self.vowel_map[c])
                i += 1
                i, diac = self._process_diacritics_only(word, n, i)
                if diac:
                    out.append(diac)
                continue
            
            # Diacritics (anuswar, chandrabindu, visarga)
            if c in self.all_diacritics:
                out.append(self.diacritic_map[c])
                i += 1
                continue
            
            # Consonant
            if c in self.all_consonants:
                base = self.base_map[c]
                i += 1
                
                # Check for halanta (्)
                if i < n and word[i] == HALANTA:
                    i += 1
                    # If followed by another consonant, it's a conjunct
                    if i < n and word[i] in self.all_consonants:
                        out.append(base)  # Bare consonant, no 'a'
                        continue
                    # Word-final halanta
                    elif i >= n:
                        out.append(base + '\\')
                        continue
                    else:
                        out.append(base)
                        continue
                
                # Check for matra (vowel sign)
                if i < n and word[i] in self.all_matras:
                    matra = self.matra_map[word[i]]
                    i += 1
                    i, diac = self._process_diacritics_only(word, n, i)
                    out.append(base + matra + (diac or ''))
                    continue
                
                # Inherent 'a' vowel (default)
                i, diac = self._process_diacritics_only(word, n, i)
                out.append(base + 'a' + (diac or ''))
                continue
            
            # Unknown character - pass through as-is
            out.append(c)
            i += 1
        
        return ''.join(out)
    
    def _process_matra_and_diacritics(self, word: str, n: int, i: int) -> Tuple[int, Optional[str]]:
        """Process matra followed by diacritics."""
        if i < n and word[i] in self.all_matras:
            matra = self.matra_map[word[i]]
            i += 1
            i, diac = self._process_diacritics_only(word, n, i)
            return i, matra + (diac or '')
        i, diac = self._process_diacritics_only(word, n, i)
        return i, diac
    
    def _process_diacritics_only(self, word: str, n: int, i: int) -> Tuple[int, Optional[str]]:
        """Process consecutive diacritics (anuswar, chandrabindu, visarga)."""
        buf = []
        while i < n and word[i] in self.all_diacritics:
            buf.append(self.diacritic_map[word[i]])
            i += 1
        return i, (''.join(buf) or None)


# ===========================================================================
# File Processing Functions
# ===========================================================================

def convert_devanagari_file(input_path: str, output_path: str = None, encoding: str = 'utf-8') -> dict:
    """Convert Devanagari file to Romanized Nepali."""
    if not os.path.exists(input_path):
        raise FileNotFoundError(f"Input file not found: {input_path}")
    
    with open(input_path, 'r', encoding=encoding) as f:
        devanagari_text = f.read()
    
    converter = DevanagariToRoman()
    romanized_text = converter.convert(devanagari_text)
    
    if output_path is None:
        base_name = Path(input_path).stem
        output_path = f"{base_name}_romanized.txt"
    
    with open(output_path, 'w', encoding=encoding) as f:
        f.write(romanized_text)
    
    return {
        'input_file': input_path,
        'output_file': output_path,
        'input_chars': len(devanagari_text),
        'output_chars': len(romanized_text),
        'input_lines': len(devanagari_text.split('\n')),
        'output_lines': len(romanized_text.split('\n')),
        'encoding': encoding
    }


def preview_conversion(input_path: str, num_lines: int = 10, encoding: str = 'utf-8') -> None:
    """Preview the first few lines of conversion."""
    converter = DevanagariToRoman()
    
    with open(input_path, 'r', encoding=encoding) as f:
        lines = f.readlines()[:num_lines]
    
    print("\n" + "=" * 80)
    print("CONVERSION PREVIEW - English words preserved / Loanwords converted")
    print("=" * 80)
    
    for i, line in enumerate(lines, 1):
        if line.strip():
            try:
                converted = converter.convert(line.strip())
                print(f"\nOriginal ({i}): {line.strip()}")
                print(f"Romanized: {converted}")
            except Exception as e:
                print(f"\nOriginal ({i}): {line.strip()}")
                print(f"Error: {e}")
    
    print("\n" + "=" * 80)


def create_sample_file(output_path: str = "sample_devanagari.txt"):
    """Create a sample Devanagari file with mixed Nepali and English loanwords."""
    sample_text = """फाइनलमा हामीले राम्रो काम गर्यौ।
The quick brown fox jumps over the lazy dog.
कम्प्युटर र इन्टरनेटको प्रयोग बढ्दो छ।
Please send me an email at test@example.com.
मोबाइल फोनको मूल्य १००० रुपैयाँ हो।
Hello, कस्तो छ? I am fine.
प्रतिशतको कर्म क्षेत्रमा काम गर्यो।
URL: https://example.com/path?query=value
बस् रहोस्, कर्म गर्।
यो होटल धेरै राम्रो छ।"""
    
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(sample_text)
    
    print(f"✓ Sample file created: {output_path}")
    print("  Contains: Regular Nepali + English words + Loanwords (फाइनल, कम्प्युटर, etc.)")
    return output_path


# ===========================================================================
# Simple Helper Functions
# ===========================================================================

def d2r(text: str) -> str:
    """Convert Devanagari string to Romanized."""
    converter = DevanagariToRoman()
    return converter.convert(text)


def convert_file_simple(input_path: str, output_path: str = None):
    """Simple one-line file conversion."""
    stats = convert_devanagari_file(input_path, output_path)
    print(f"✅ Converted: {stats['input_file']} → {stats['output_file']}")
    print(f"   {stats['input_chars']} characters → {stats['output_chars']} characters")
    return stats


# ===========================================================================
# Initialize and Display
# ===========================================================================

print("=" * 80)
print("✅ NEPALI DEVANAGARI TO ROMANIZED CONVERTER")
print("   Fixed Recursion Error - Now Working Properly")
print("=" * 80)
print("\n🔍 FEATURES:")
print("  • Detects pure English words → leaves them unchanged")
print("  • Detects English loanwords in Devanagari (फाइनल → final)")
print("  • Preserves URLs, emails, numbers, punctuation")
print("  • Handles mixed Nepali-English text")
print("\n📝 EXAMPLES:")
print("  फाइनलमा → finalma")
print("  कम्प्युटर → computer")
print("  The quick brown fox → The quick brown fox (unchanged)")
print("\n" + "=" * 80)

print("\n💡 QUICK START:")
print("  1. Create a sample file: create_sample_file()")
print("  2. Preview conversion: preview_conversion('sample_devanagari.txt')")
print("  3. Convert a file: convert_file_simple('sample_devanagari.txt')")
print("  4. Convert a string: d2r('फाइनलमा कम्प्युटर छ')")
print("=" * 80)

✅ NEPALI DEVANAGARI TO ROMANIZED CONVERTER
   Fixed Recursion Error - Now Working Properly

🔍 FEATURES:
  • Detects pure English words → leaves them unchanged
  • Detects English loanwords in Devanagari (फाइनल → final)
  • Preserves URLs, emails, numbers, punctuation
  • Handles mixed Nepali-English text

📝 EXAMPLES:
  फाइनलमा → finalma
  कम्प्युटर → computer
  The quick brown fox → The quick brown fox (unchanged)


💡 QUICK START:
  1. Create a sample file: create_sample_file()
  2. Preview conversion: preview_conversion('sample_devanagari.txt')
  3. Convert a file: convert_file_simple('sample_devanagari.txt')
  4. Convert a string: d2r('फाइनलमा कम्प्युटर छ')


In [28]:
create_sample_file()

✓ Sample file created: sample_devanagari.txt
  Contains: Regular Nepali + English words + Loanwords (फाइनल, कम्प्युटर, etc.)


'sample_devanagari.txt'

In [29]:
preview_conversion('sample_devanagari.txt')


CONVERSION PREVIEW - English words preserved / Loanwords converted

Original (1): फाइनलमा हामीले राम्रो काम गर्यौ।
Romanized: finalma haamiile raamro kaama garyau।

Original (2): The quick brown fox jumps over the lazy dog.
Romanized: The quick brown fox jumps over the lazy dog.

Original (3): कम्प्युटर र इन्टरनेटको प्रयोग बढ्दो छ।
Romanized: computer ra internetko prayoga baDhdo chha।

Original (4): Please send me an email at test@example.com.
Romanized: Please send me an email at test@example.com.

Original (5): मोबाइल फोनको मूल्य १००० रुपैयाँ हो।
Romanized: mobile phonako muulya १००० rupaiyaa** ho।

Original (6): Hello, कस्तो छ? I am fine.
Romanized: Hello, kasto chha? I am fine.

Original (7): प्रतिशतको कर्म क्षेत्रमा काम गर्यो।
Romanized: pratishatako karma kshetramaa kaama garyo।

Original (8): URL: https://example.com/path?query=value
Romanized: URL: https://example.com/path?query=value

Original (9): बस् रहोस्, कर्म गर्।
Romanized: bas\ rahos, karma gar।

Original (10): यो होट

In [30]:
convert_devanagari_file('devnagari.txt')

{'input_file': 'devnagari.txt',
 'output_file': 'devnagari_romanized.txt',
 'input_chars': 739,
 'output_chars': 937,
 'input_lines': 5,
 'output_lines': 5,
 'encoding': 'utf-8'}

In [33]:
import ntr
nepali_corpus = '''
वैशाख २१ – आर्सनललाई हराउँदै एथ्लेटिको मड्रिड युरोपा लिगको फाइनलमा प्रवेश गरेको छ ।
घरेलु मैदानमा भएको च्याम्पियन्स लिगको दोस्रो लेगमा एथ्लेटिको मड्रिडले आर्सनललाई एक शून्यले हराउँदै समग्रमा दुई एकको अग्रताका साथ फाइनलमा प्रवेश गरेको हो ।
सन् २०१६ को च्याम्पियन्स लिगको फाइनलमा रियल मड्रिडसँग पराजित भएपछि एथ्लेटिको पहिलो पटक युरोपियन फुटबल प्रतियोगिताको फाइनलमा पुगेको हो ।
प्रशिक्षक विङगरले युरोपियन खेलमा सफलता हात पार्न सकेनन् । प्रिमियर लिगमा राम्रो प्रदर्शन निकाल्न नसकेपछि प्रशिक्षकका रुपमा उनको ठूलो आलोचना भएको थियो । जसका कारण उनले इमिरेट्स छोड्ने निर्णयमा पुगेका थिए ।
आर्सनलको प्रशिक्षकका रुपमा उनले तीन वटा प्रिमियर र सात वटा एफ ए कपको उपाधि जिते पनि युरोपियन प्रतियोगितामा भने २१ सिजनमा दुई पटक मात्र फाइनल प्रवेश गरेका छन् ।
'''
print(ntr.nep_to_rom(nepali_corpus))

 
 waishakh 21 – aarsanalalai haraundai ethletiko madrid yuropa ligako phainalama prawesh gareko chha . 
 gharelu maidanama bhaeko chyampiyansa ligako dosro legama ethletiko madridale aarsanalalai eka shunyale haraundai samagrama dui ekako agrataka sath phainalama prawesh gareko ho . 
 san 2016 ko chyampiyansa ligako phainalama riyal madridasang parajit bhaepachhi ethletiko pahilo patak yuropiyan phutabal pratiyogitako phainalama pugeko ho . 
 prashikshak winagarale yuropiyan khelama saphalata hat parna sakenan . primiyar ligama ram्ro pradarshan nikalna nasakepachhi prashikshakaka rupama unako thulo aalochana bhaeko thiyo . jasaka karan unale imiretsa chhodne nirnayama pugeka thie . 
 aarsanalako prashikshakaka rupama unale tin wata primiyar ra sat wata epha e kapako upadhi jite pani yuropiyan pratiyogitama bhane 21 sijanama dui patak matra phainal prawesh gareka chhan . 



In [34]:
ntr

<module 'ntr' from '/home/lang-chain/Documents/Astra_agentic_RAG/.venv/lib/python3.11/site-packages/ntr/__init__.py'>